# Lab 9 — Fine-tuning a Small Language Model on Crash Narratives

**AI4Mobility (CGN 6933) · Module 6 · reads with 6.1 (the adaptation ladder) and 2.13 (does it earn its keep?)**

In 2.13 a TF-IDF bag of words with a logistic regression beat every frozen BERT probe on
crash narratives, and a fine-tuned DistilBERT did not close the gap. Section 6.1 then gave you a
ladder for adapting a model: **prompt it, then give it retrieved examples, then train a small
adapter (LoRA), and only then think about anything bigger.** This lab climbs the first three
rungs on the same 399 Florida crash reports, with a 494-million-parameter open model
that fits on the free Colab GPU, and asks one question at every rung: *did that beat the bag of
words, on a split you can defend?*

| | |
|---|---|
| **What you build** | A narrative-to-JSON extractor for six crash fields, four ways: prompt only, fixed examples, retrieved examples, and a LoRA adapter |
| **What it proves** | Whether LoRA beats prompting and retrieval on a split you can defend, and whether any of them beats a bag of words |
| **Check it against** | The baseline, named now: **TF-IDF + logistic regression**, the 2.13 winner, on the same corridor-grouped folds. Also the majority class, and DistilBERT from 2.13 |
| **Runs on** | Colab free tier with a **T4 GPU** (Runtime → Change runtime type). Sections 1–3 run on CPU in seconds |
| **Needs** | `transformers`, `peft`, `accelerate`, `scikit-learn` (one setup cell) · about 1 GB of model download · the narrative file from Canvas · **no API key** |

---

### The honest disclosure, before anything else

**This is not a sample of Florida crashes.** The 399 reports were chosen from
20 corridors in Pasco, Hernando and Pinellas counties to study left turns and
pedestrians, so 245 of them are left-turn crashes. No number in this
notebook is a statement about Florida. Every number is a statement about *these reports*, scored
on corridors the model never saw.

The answers you score against are the **coded fields** of the same police report (the boxes the
officer ticked), not a reading of the narrative. Section 1 shows that some of those boxes are
simply never mentioned in the text, and drops them rather than pretending a model could read them.

**The narratives are not in this notebook.** They are redacted police text shared with enrolled
students through Canvas. Keep them out of public repos and hosted chatbots, and clear outputs
before you share your copy.

## 0 · Setup

Two cells. The first installs; the second is the **only** place the data path appears.

**Getting the data.** Download `crash_narratives_joined.json` from the AI4Mobility Canvas course. That
is the only place it is distributed; there is no public URL. In Colab, run cell 2 and upload the file
when the picker appears, or copy it to your Drive and set `DATA_PATH` to point at it.

In [ ]:
# ===========================================================================
#  CELL 1 — install and check the runtime. Safe to re-run.
# ===========================================================================
import subprocess, sys
# transformers>=4.56 for the `dtype=` argument used below; peft for LoRA. No API key anywhere.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "transformers>=4.56", "peft>=0.17", "accelerate", "scikit-learn"], check=True)

import torch, transformers, peft, sklearn
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# A T4 has no fast bfloat16, so on the GPU we run the 0.5B model in float32 (2 GB, fits easily).
# On a CPU with bfloat16 support, bfloat16 halves the memory and roughly doubles the speed.
DTYPE = torch.float32 if DEVICE == "cuda" else torch.bfloat16
torch.manual_seed(0)
print("torch", torch.__version__, "· transformers", transformers.__version__, "· peft", peft.__version__)
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only", "· dtype:", DTYPE)
if DEVICE == "cpu":
    print("No GPU. In Colab: Runtime > Change runtime type > T4 GPU. Sections 1-3 are fine on CPU; "
          "sections 4-6 will be slow (see the timing table in section 7).")

In [ ]:
# ===========================================================================
#  CELL 2 — load the narratives. ONE place to point at the file.
# ===========================================================================
import os, json, urllib.request

# The narratives are course data, distributed ONLY through Canvas. There is no public copy and no URL.
#   1. Download crash_narratives_joined.json from the AI4Mobility Canvas course.
#   2. Either upload it when the file picker appears below, or put it somewhere and set DATA_PATH.
DATA_PATH = "crash_narratives_joined.json"   # e.g. "/content/drive/MyDrive/ai4mobility/crash_narratives_joined.json"
DATA_URL  = ""                               # leave empty: the course does not host this file publicly

if not os.path.exists(DATA_PATH):
    if DATA_URL:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    else:
        try:
            from google.colab import files           # Colab: a file picker appears below this cell
            print("Upload crash_narratives_joined.json (download it from Canvas first)")
            DATA_PATH = next(iter(files.upload()))
        except ImportError:
            raise FileNotFoundError("Download crash_narratives_joined.json from Canvas, put it next to "
                                    "this notebook, or set DATA_PATH above.")

D = json.load(open(DATA_PATH))
TEXTS  = [r["text"] for r in D]
GROUPS = [r["corridor"] for r in D]
print(f"{len(D)} reports · {len(set(GROUPS))} corridors · "
      f"median narrative {sorted(len(t.split()) for t in TEXTS)[len(D)//2]} words")
# These are redacted police narratives shared for this course only. Do not paste them into a
# public repo, a forum, or a hosted chatbot, and clear this notebook's outputs before you share it.

## 1 · Six fields a reader could actually fill in

The coded report has 33 fields. A narrative-to-JSON extractor is only fair to score on fields the
narrative *could* tell you. We keep six:

| Field | Values | From the coded field | Count of the rarer side |
|---|---|---|---|
| `crash_type` | left_turn · angle · pedestrian_bicycle · other | simplified crash type (rear end and the rest → other) | other: 16 |
| `pedestrian` | yes · no | pedestrian involved | yes: 36 |
| `vehicles` | 1 · 2 · 3+ | total vehicles | 1: 36, 3+: 37 |
| `injury` | yes · no | severity is anything but No Injury | no: 174 |
| `hit_and_run` | yes · no | hit and run | yes: 37 |
| `distracted` | yes · no | distracted driving | yes: 89 |

Together these cover the four 2.13 targets (distracted, pedestrian, left turn, injury) plus two that
an agency would want from the same text.

In [ ]:
# ===========================================================================
#  CELL 3 — the schema: six fields, each read off a coded field of the same report.
# ===========================================================================
import re, collections, numpy as np

def crash_type(r):
    return {"Left Turn": "left_turn", "Angle": "angle", "Pedestrian": "pedestrian_bicycle",
            "Bicycle": "pedestrian_bicycle"}.get(r["S4_CRASH_TYPE_SIMPLIFIED"], "other")
def vehicles(r):
    k = int(r["TOTAL_NUMBER_OF_VEHICLES"]); return "3+" if k >= 3 else str(k)
yn = lambda b: "yes" if b else "no"

FIELDS = {   # field -> (allowed values, how the answer key is read off the coded report)
 "crash_type":  (["left_turn", "angle", "pedestrian_bicycle", "other"], crash_type),
 "pedestrian":  (["yes", "no"], lambda r: yn(r["S4_IS_PEDESTRIAN_INVOLVED"] == "Y")),
 "vehicles":    (["1", "2", "3+"], vehicles),
 "injury":      (["yes", "no"], lambda r: yn(r["S4_CRASH_SEVERITY"] != "No Injury")),
 "hit_and_run": (["yes", "no"], lambda r: yn(r["S4_IS_HIT_AND_RUN"] == "Y")),
 "distracted":  (["yes", "no"], lambda r: yn(r["S4_IS_DISTRACTED"] == "Y")),
}
GOLD = [{f: fn(r) for f, (_, fn) in FIELDS.items()} for r in D]
gold_json = lambda i: json.dumps(GOLD[i])

for f in FIELDS:
    print(f"{f:12s}", dict(collections.Counter(g[f] for g in GOLD).most_common()))

In [ ]:
# ===========================================================================
#  CELL 4 — one scorer for every method. A missing or invalid value counts as WRONG.
# ===========================================================================
from sklearn.metrics import f1_score, precision_score, recall_score

def score(preds, idx):
    out = {}
    for f, (vals, _) in FIELDS.items():
        g = [GOLD[i][f] for i in idx]
        p = [(preds[i] or {}).get(f) or "__missing__" for i in idx]
        if vals == ["yes", "no"]:
            gy, py_ = [x == "yes" for x in g], [x == "yes" for x in p]
            out[f] = {"F1": f1_score(gy, py_, zero_division=0),
                      "P": precision_score(gy, py_, zero_division=0), "R": recall_score(gy, py_, zero_division=0)}
        else:
            out[f] = {"F1": f1_score(g, p, labels=vals, average="macro", zero_division=0)}   # macro-F1
    out["mean"] = {"F1": float(np.mean([out[f]["F1"] for f in FIELDS]))}
    return out

SCORES = {}
def show(name, s):
    SCORES[name] = s
    print(f"{name:34s}" + "".join(f"{f[:11]:>12s}" for f in list(FIELDS) + ["mean"]))
    print(" " * 34 + "".join(f"{s[f]['F1']:12.3f}" for f in list(FIELDS) + ["mean"]))

### Fields we dropped, and why

Before you ask any model to read a field out of text, check that the text contains it. The audit
below uses a deliberately generous regular expression for each field and counts how many of the
coded positives mention it at all. That share is a **ceiling**: no reader, human or model, can
recover a fact the officer did not write down.

In [ ]:
# ===========================================================================
#  CELL 5 — availability audit: is the field even WRITTEN in the narrative?
# ===========================================================================
AUDIT = {   # field: (coded positive, a generous regex for "the officer wrote about it")
 "weather = rain":  (lambda r: r["WEATHER_CONDITION"] == "Rain", r"rain|wet|shower|storm|drizzl|downpour"),
 "light = dark":    (lambda r: r["LIGHT_CONDITION"].startswith("Dark"), r"\bdark|night|street ?light|headlight|lighting"),
 "surface = wet":   (lambda r: r["ROAD_SURFACE_CONDITION"] == "Wet", r"\bwet|rain|slick|hydroplan|standing water"),
 "alcohol":         (lambda r: r["S4_IS_ALCOHOL_RELATED"] == "Y", r"alcohol|impair|\bDUI|\bBAC\b|breath|intoxicat|beer|drunk|sobriety"),
 "hit_and_run":     (lambda r: r["S4_IS_HIT_AND_RUN"] == "Y", r"fled|left the scene|leave the scene|unknown (?:vehicle|driver)|did not stop|failed to remain"),
 "pedestrian":      (lambda r: r["S4_IS_PEDESTRIAN_INVOLVED"] == "Y", r"pedestrian|non-?motorist|NM ?0?\d|walking|crosswalk"),
}
print(f"{'field':16s}{'coded yes':>10s}{'...mentioned':>14s}{'ceiling':>9s}")
for name, (pos, rx) in AUDIT.items():
    P = [r for r in D if pos(r)]
    hit = sum(bool(re.search(rx, r["text"], re.I)) for r in P)
    print(f"{name:16s}{len(P):10d}{hit:14d}{100*hit/max(1,len(P)):8.1f}%")

| Field | Coded yes | Narrative mentions it | Ceiling | Decision |
|---|---|---|---|---|
| weather = rain | 16 | 2 | 12.5% | **dropped**: this is the 12.5% ceiling from 2.13; weather belongs to a weather feed, not a narrative |
| light = dark | 119 | 6 | 5.0% | **dropped**: officers almost never describe the lighting |
| surface = wet | 27 | 5 | 18.5% | **dropped** for the same reason |
| alcohol | 9 | 7 | 77.8% | **dropped**: readable, but 9 positives across 5 folds cannot support an F1 |
| hit_and_run | 37 | 23 | 62.2% | **kept**, with a known ceiling (section 9) |
| pedestrian | 36 | 35 | 97.2% | **kept**: nearly always written |

Notice what a careless schema would have done. Put `light_condition` in the JSON and every method
"learns" it from the base rate (119 of
399 are dark), and the score looks respectable while measuring nothing about reading.

> **Your turn — exercise 1.** The regex for hit-and-run misses some phrasings. Read the coded field
> definition, add two phrasings you think officers use, and re-run the audit. Does the ceiling move
> by more than a couple of reports? What does that tell you about where the remaining misses come from?

## 2 · The baseline, named up front

**TF-IDF + logistic regression** with the exact 2.13 settings: English stop words, sublinear term
frequency, words that appear in at least two reports, single words and pairs, classes reweighted
by their frequency. One small model per field. It is the number every rung of the ladder has to beat.

We score each yes/no field with **F1** (it ignores the easy "no" reports), and the two multi-class
fields with **macro-F1** (the average F1 across classes, so `other` counts as much as `left_turn`).
A missing or invalid value counts as wrong. That last rule matters later: a language model that
writes broken JSON gets no partial credit.

In [ ]:
# ===========================================================================
#  CELL 6 — the split: five folds, grouped by corridor. Same folds as 2.13.
# ===========================================================================
from sklearn.model_selection import GroupKFold
FOLDS = list(GroupKFold(n_splits=5).split(np.arange(len(D)), None, GROUPS))
for k, (tr, te) in enumerate(FOLDS):
    assert not set(GROUPS[i] for i in tr) & set(GROUPS[i] for i in te)   # no corridor on both sides
    print(f"fold {k}: test {len(te):3d} reports from {len(set(GROUPS[i] for i in te))} corridors")
ALL = np.arange(len(D))

In [ ]:
# ===========================================================================
#  CELL 7 — the baseline, named up front: majority class, then TF-IDF + logistic regression
#  with exactly the settings that won section 2.13.
# ===========================================================================
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def tfidf():
    return TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2, ngram_range=(1, 2))

def classical(folds):
    pred = [dict() for _ in D]
    for tr, te in folds:
        v = tfidf(); Xtr = v.fit_transform([TEXTS[i] for i in tr]); Xte = v.transform([TEXTS[i] for i in te])
        for f in FIELDS:
            m = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, [GOLD[i][f] for i in tr])
            for i, p in zip(te, m.predict(Xte)): pred[i][f] = p
    return pred

maj = [None] * len(D)
for tr, te in FOLDS:
    mc = {f: collections.Counter(GOLD[i][f] for i in tr).most_common(1)[0][0] for f in FIELDS}
    for i in te: maj[i] = mc
show("majority class", score(maj, ALL))

t0 = time.time(); PRED = {"tfidf_lr": classical(FOLDS)}
show("TF-IDF + logistic regression", score(PRED["tfidf_lr"], ALL))
print(f"\n(all six fields, all five folds, in {time.time()-t0:.0f} s)")

On all five corridor folds:

| | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted |
|---|---|---|---|---|---|---|
| majority class | 0.190 | 0.000 | 0.300 | 0.721 | 0.000 | 0.000 |
| **TF-IDF + LR** | **0.578** | **0.882** | **0.886** | **0.707** | **0.688** | **0.590** |

The whole baseline, six fields by five folds, trains in about 63 seconds on a CPU.

**Tie-back to 2.13.** The same model on the same folds gives distracted-driving average precision
**0.614** here against the **0.619**
quoted in 2.13. The small difference comes from library versions, not from a different split. DistilBERT
reached 0.488 on the same target after 35.1 minutes of training.

## 3 · A split you can defend

Every report belongs to one of 20 corridors, and reports from the same
corridor share street names, intersection layouts, and often the same officer's writing habits. If
reports from one corridor land on both sides of the split, the model can pass the test by
recognising the corridor instead of reading the crash. So the five folds are **grouped by
corridor**: every test report comes from a corridor the model has never seen. These are exactly
the folds used in 2.13, so the numbers line up.

What would the shortcut be worth? Shuffle the reports, ignore corridors, and fit the same baseline:

In [ ]:
# ===========================================================================
#  CELL 8 — the split you cannot defend: shuffle the reports, ignore corridors.
# ===========================================================================
from sklearn.model_selection import KFold
shuffled = list(KFold(5, shuffle=True, random_state=0).split(ALL))
show("TF-IDF + LR, RANDOM split", score(classical(shuffled), ALL))
show("TF-IDF + LR, corridor split", SCORES["TF-IDF + logistic regression"])

| TF-IDF + LR | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted |
|---|---|---|---|---|---|---|
| random split | 0.722 | 0.899 | 0.823 | 0.701 | 0.623 | 0.562 |
| corridor split | 0.578 | 0.882 | 0.886 | 0.707 | 0.688 | 0.590 |

The honest reading is **mixed**, and that is worth saying plainly. On `crash_type` the random split
is +0.144 macro-F1 better, which is the corridor shortcut. Corridors differ in their crash
mix and their geometry words, and a shuffled split lets the model use that. On the other five fields
the gaps run both ways (the largest is 0.065), and the six-field means come out at
0.722 for the random split and 0.722 for the corridor split. With 36 pedestrian and 37 hit-and-run positives, a few
reports moving fold is enough to do that. So the random split does not inflate everything. It inflates
the one field that depends on place, and you would not know which field that was without running both.

Retrieval has its own version of this leak, and section 6 measures it: the examples you paste into
the prompt must come from **training corridors only**.

> **Going deeper — why not stratify too?** `GroupKFold` balances fold *sizes*, not label mixes. With
> 8 rear-end crashes in 399, one fold can get most of them. `StratifiedGroupKFold` tries to balance
> both. We keep plain `GroupKFold` so the folds match 2.13 exactly. Exercise 6 asks you to swap it
> and see which numbers move.

## 4 · The ladder, in plain words

Section 6.1 ordered the ways to adapt a model by cost. This lab climbs the first three rungs with the
model held fixed, so the only thing that changes is how much you teach it.

| Rung | What changes | What you pay |
|---|---|---|
| **Prompt** | only the instructions | nothing but inference |
| **Few-shot / retrieval** | the prompt also carries worked examples | longer prompts, so slower inference; a retrieval index |
| **LoRA** | a small set of new weights, trained on your labelled reports | a training run per fold; an adapter file to manage |

The model is **Qwen2.5-0.5B-Instruct**: 494 million parameters, open weights (Apache-2.0), already
tuned to follow chat instructions. It is deliberately small. It fits on a free T4 with room for
training, and it is the size an agency could run on its own hardware without sending a police
report to anyone's API.

**The equations, in words.**

| Symbol | In plain words |
|---|---|
| $W' = W + \tfrac{\alpha}{r}\,B A$ | Keep every original weight matrix $W$ frozen. Learn a correction that is the product of two thin matrices: $A$ squeezes the input down to $r$ numbers, and $B$ expands them back. $r$ is how thin, and $\alpha / r$ is a volume knob on the correction. |
| $r = 16$ on $q, k, v, o$ | We add the correction only to the four attention projections in each of the 24 blocks. That is **2.16 M trainable numbers, 0.44% of the model**. |
| $\mathcal{L} = -\tfrac{1}{T}\sum_t \log p(y_t \mid \text{prompt}, y_{<t})$ | The training loss is how surprised the model is by each token of the *correct JSON*, averaged over the answer. The prompt and narrative tokens are not scored; the model only learns to answer. |
| $\text{sim}(q, d) = \cos(\text{tfidf}(q), \text{tfidf}(d))$ | Retrieval picks the training reports whose word profile points the same way as the new report's. It uses the baseline's own features, so what you are testing is how the examples are *used*, not how they are found. |

> **Going deeper — why low rank is enough.** Fine-tuning moves the 896 × 896 query matrix in one block by some
> change $\Delta W$. LoRA's bet is that the useful part of $\Delta W$ for one narrow task lives in a few
> directions, so $\Delta W \approx BA$ with $B$ being 896 × 16 and $A$ 16 × 896: 28,672 numbers instead of
> 802,816. $B$ starts at zero, so at step 0 the adapted model *is* the original model, and training
> can only move it away as far as the data pushes. Exercise 5 asks you to change $r$.

## 5 · Rung 1 — just ask

The prompt is the schema, written out with the allowed values, and the narrative. Decoding is
greedy, so the same input always gives the same output, and generation stops after 80 tokens. The
parser is strict on purpose: it takes the first `{...}` in the reply, maps `3` to `3+` and
`true` to `yes`, and marks any field that is missing or outside its allowed values as invalid.

In [ ]:
# ===========================================================================
#  CELL 9 — the small open model, the prompt, and a strict parser.
# ===========================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # 494M parameters, Apache-2.0, about 1 GB download
tok = AutoTokenizer.from_pretrained(MODEL_ID)
def load_model():
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()
model = load_model()

SPEC = """Read the police crash narrative and fill in this JSON record. Use only these values:
crash_type: "left_turn" (a vehicle turning left), "angle" (vehicles on crossing paths, no left turn), "pedestrian_bicycle" (a pedestrian or cyclist was struck), or "other"
pedestrian: "yes" if a pedestrian was involved, else "no"
vehicles: "1", "2", or "3+" (motor vehicles involved)
injury: "yes" if anyone was injured or killed, else "no"
hit_and_run: "yes" if a driver left the scene, else "no"
distracted: "yes" if a driver was distracted or inattentive, else "no"
Answer with the JSON object only."""

def messages(text, shots=()):
    m = [{"role": "system", "content": SPEC}]
    for t, a in shots:                              # worked examples: a narrative and its answer
        m += [{"role": "user", "content": "Narrative:\n" + t}, {"role": "assistant", "content": a}]
    return m + [{"role": "user", "content": "Narrative:\n" + text}]

@torch.no_grad()
def generate(m, msgs, max_new=80):
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True).to(DEVICE)
    out = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def parse(s):
    """-> (record or None, parsed as JSON?, every field present with an allowed value?)"""
    mm = re.search(r"\{.*?\}", s, re.S)
    if not mm: return None, False, False
    try: d = json.loads(mm.group(0))
    except Exception: return None, False, False
    if not isinstance(d, dict): return None, False, False
    rec, ok = {}, True
    for f, (vals, _) in FIELDS.items():
        v = str(d.get(f, "")).strip().lower()
        v = {"true": "yes", "false": "no"}.get(v, v)
        if f == "vehicles" and v.isdigit(): v = "3+" if int(v) >= 3 else v
        if v not in vals: ok, v = False, None
        rec[f] = v
    return rec, True, ok

print(f"{sum(p.numel() for p in model.parameters()):,} parameters on {DEVICE}")
print("prompt tokens, system part only:", len(tok(SPEC)["input_ids"]))

**How long this takes.** On the build machine (8 CPU threads, bfloat16, no GPU) each
narrative took about 2.57 s zero-shot and
1.96 s with three examples in the prompt.
On a T4 it should be faster. **TODO-after-run**: time it on a T4 and put the number here. Set
`RUN_FOLDS = [0]` below for a one-fifth run if you are short on time; every printout says which
folds it covers.

In [ ]:
# ===========================================================================
#  CELL 10 — how much to run. The whole lab is 5 folds; one fold is a fair quick look.
# ===========================================================================
RUN_FOLDS = [0, 1, 2, 3, 4]      # set to [0] for a ~1/5-size run; every table then says which folds it covers
EVAL = np.concatenate([FOLDS[k][1] for k in RUN_FOLDS])
VALID, SECS = {}, {}

def run_prompting(name, shots_for):
    """shots_for(train_idx) -> function(i) returning the worked examples for test report i."""
    pred, ok_json, ok_schema, t0 = [None] * len(D), [], [], time.time()
    for k in RUN_FOLDS:
        tr, te = FOLDS[k]; pick = shots_for(tr)
        for i in te:
            rec, vj, vs = parse(generate(model, messages(TEXTS[i], pick(i))))
            pred[i] = rec; ok_json.append(vj); ok_schema.append(vs)
        print(f"  {name}: fold {k} done, {time.time()-t0:.0f} s")
    PRED[name], VALID[name], SECS[name] = pred, (np.mean(ok_json), np.mean(ok_schema)), (time.time()-t0)/len(EVAL)
    print(f"valid JSON {100*np.mean(ok_json):.1f}% · every field valid {100*np.mean(ok_schema):.1f}% · "
          f"{SECS[name]:.2f} s per narrative")
    show(name, score(pred, EVAL))
    show("TF-IDF + LR, same folds", score(PRED["tfidf_lr"], EVAL))

In [ ]:
# ===========================================================================
#  CELL 11 — rung 1: zero-shot. The schema, the narrative, nothing else.
# ===========================================================================
run_prompting("zero-shot", lambda tr: (lambda i: ()))

**What you should see.** Valid JSON almost every time (99.5%), and almost
nothing inside it. The 0.5B model wrote `left_turn` for 394 of
399 reports and `3+` vehicles for 367. It answered
hit-and-run "yes" 4 times, and in
108 replies it renamed or dropped the `pedestrian`
key. Only 71% of replies had all six fields valid. It is reading the *schema*,
not the narrative: the first allowed value of a list is a strong pull for a model this small. `injury`,
with F1 0.623, is the only field where it does better than the majority class
(0.721 by saying "yes" to everything), and only barely.

Valid JSON is not a measure of correct JSON. Report both.

## 6 · Rung 2 — show it worked examples

Two ways to fill the prompt with three solved reports. **Fixed**: the same three for every
narrative (the shortest training report of each main crash type). **Retrieved**: the three training
reports most similar to the one being read, found with the baseline's TF-IDF features. The index is
rebuilt inside every fold from the **training corridors only**.

In [ ]:
# ===========================================================================
#  CELL 12 — rung 1b: three fixed worked examples, the shortest training report of each main type.
# ===========================================================================
def fixed_shots(tr):
    shots = []
    for ct in ["left_turn", "angle", "pedestrian_bicycle"]:
        j = min([i for i in tr if GOLD[i]["crash_type"] == ct], key=lambda i: len(TEXTS[i]))
        shots.append((TEXTS[j], gold_json(j)))
    return lambda i: shots
run_prompting("few-shot, fixed 3", fixed_shots)

In [ ]:
# ===========================================================================
#  CELL 13 — rung 2: retrieval. Three most similar TRAINING-corridor reports as the examples.
# ===========================================================================
def retrieved_shots(tr, k=3):
    tr = np.array(tr); v = tfidf(); X = v.fit_transform([TEXTS[i] for i in tr])   # fit on training corridors only
    def pick(i):
        sim = (X @ v.transform([TEXTS[i]]).T).toarray().ravel()
        top = tr[np.argsort(-sim)[:k]][::-1]                 # most similar sits nearest the question
        return [(TEXTS[j], gold_json(j)) for j in top]
    return pick
run_prompting("few-shot, retrieved 3", retrieved_shots)

| | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted | mean | all fields valid |
|---|---|---|---|---|---|---|---|---|
| zero-shot | 0.191 | 0.000 | 0.072 | 0.623 | 0.049 | 0.154 | 0.181 | 71% |
| fixed 3 | 0.370 | 0.262 | 0.585 | 0.721 | 0.053 | 0.042 | 0.339 | 99% |
| retrieved 3 | 0.492 | 0.667 | 0.686 | 0.697 | 0.327 | 0.528 | **0.566** | 95% |
| *TF-IDF + LR* | 0.578 | 0.882 | 0.886 | 0.707 | 0.688 | 0.590 | *0.722* | — |

Fixed examples teach the **format**: the share of fully valid replies jumps to 99%,
and the model starts using all the values. They do not teach the **task**. It now calls
109 reports pedestrian crashes where there are 36,
and it still answers hit-and-run "yes" only 1 time(s).

Retrieved examples are where the signal shows up. Mean F1 goes from 0.339 to
**0.566**, and the pedestrian F1 goes from 0.262 to
0.667. Be honest about *why*. Three neighbours chosen by TF-IDF usually
share the answer, so a model that copies the most similar example's labels already gets a long way. Part
of what retrieval buys here is a nearest-neighbour classifier, run through a language model. Exercise 3
measures how much.

Against the bag of words it trails on all six fields, and it costs about 1.96 s
per report against milliseconds.

### The retrieval leak

Now do it wrong once on purpose. Let retrieval search **every other report**, including those from the
test report's own corridor. This is the mistake that happens by default when the index is built once,
before the split.

In [ ]:
# ===========================================================================
#  CELL 14 (optional, ~same time as cell 13) — the leak: retrieve from EVERY other report,
#  including the test report's own corridor. Run it once, so you know what it looks like.
# ===========================================================================
def leaky_shots(tr, k=3):
    def pick(i):
        others = [j for j in range(len(D)) if j != i]
        return retrieved_shots(others, k)(i)
    return pick
run_prompting("retrieved 3, LEAKY", leaky_shots)

| retrieval from | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted | mean |
|---|---|---|---|---|---|---|---|
| training corridors only | 0.492 | 0.667 | 0.686 | 0.697 | 0.327 | 0.528 | 0.566 |
| every other report (leaky) | 0.550 | 0.743 | 0.636 | 0.737 | 0.304 | 0.583 | 0.592 |

The leak raises the mean from 0.566 to 0.592 (+0.026). It is not uniform:
it helps `crash_type` (+0.058), `pedestrian` (+0.076), `injury` (+0.040), `distracted` (+0.055), and it hurts `vehicles` (-0.050), `hit_and_run` (-0.023). Neighbours from the same corridor
share its crash mix and its street vocabulary. That helps the fields that depend on place and does
nothing for counting vehicles.

Compare section 3. There, shuffling the reports moved the TF-IDF six-field mean by 0.000. Here, the
leak moves retrieval's mean by +0.026. A language model that copies its examples is **more** exposed
to a leaky index than a bag of words is to a leaky split, because the leaked neighbour sits right in the
prompt. On five folds and 399 reports, a few hundredths is roughly the size a handful of
reports can move a field, so do not read the exact figure too closely. Read the direction, and read that
the leaky row is a number you could not quote to an agency that wants this on a corridor nobody labelled.

Leaky retrieval also ran slower on the build machine (9.14 s per report against
1.96 s). That reflects a busier shared CPU, not the method: the prompts are the same length.

> **Your turn — exercise 2.** Change `k` in `retrieved_shots` to 1 and to 6. More examples make a
> longer prompt. Does the score keep rising, and what happens to seconds per report?
>
> **Your turn — exercise 3.** Build the "copy the nearest neighbour" baseline with no language model:
> for each test report, output the gold JSON of the single most similar training report. Score it with
> `score()`. How much of the retrieval rung's gain does it explain?

## 7 · Rung 3 — LoRA

One adapter per fold, trained on that fold's four training folds (about 320 reports), then asked the
**plain zero-shot prompt** about the held-out corridors. There are no examples in the prompt, so
whatever it gets right it learned from training.

Settings: rank 16, alpha 32, dropout 0.05, on the q/k/v/o projections; AdamW at 1e-4 (fp32 adapter, non-finite steps skipped) with a 10-step
warm-up and a linear decay; 2 epochs; 4 reports per update. Nothing was tuned on the test folds.
If you tune them, tune them inside the training folds.

In [ ]:
# ===========================================================================
#  CELL 15 — rung 3: LoRA. Freeze all 494M weights; learn a thin correction on the attention layers.
# ===========================================================================
from peft import LoraConfig, get_peft_model

def lora_train(train_idx, r=16, alpha=32, epochs=2, lr=1e-4, accum=4, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    pm = get_peft_model(load_model(), LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.05, task_type="CAUSAL_LM",
                                                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    params = [p for p in pm.parameters() if p.requires_grad]
    for p in params: p.data = p.data.float()     # the adapter itself always trains in fp32
    opt = torch.optim.AdamW(params, lr=lr)
    total = epochs * int(np.ceil(len(train_idx) / accum))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 10) * max(0.0, 1 - s / total))
    inner, losses = pm.base_model.model, []
    pm.train()
    for ep in range(epochs):
        order = rng.permutation(train_idx)
        for b in range(0, len(order), accum):
            for i in order[b:b + accum]:
                p = tok.apply_chat_template(messages(TEXTS[i]), add_generation_prompt=True, tokenize=False)
                pi = tok(p, add_special_tokens=False)["input_ids"]
                ai = tok(gold_json(i) + "<|im_end|>", add_special_tokens=False)["input_ids"]
                ids = torch.tensor([pi + ai], device=DEVICE)
                # The loss counts ONLY the answer tokens. We also project only those positions to the
                # 151,936-word vocabulary, which is what keeps this inside a free-tier memory budget.
                h = inner.model(input_ids=ids).last_hidden_state[:, len(pi) - 1:-1]
                loss = torch.nn.functional.cross_entropy(inner.lm_head(h).float()[0], ids[0, len(pi):])
                if not torch.isfinite(loss): continue      # guard: one bad step must not poison the adapter
                (loss / accum).backward(); losses.append(loss.item())
            if torch.isfinite(torch.nn.utils.clip_grad_norm_(params, 1.0)): opt.step()
            sched.step(); opt.zero_grad()
        print(f"    epoch {ep+1}: mean answer loss {np.mean(losses[-len(train_idx):]):.3f}")
    pm.print_trainable_parameters()
    return pm.eval()

In [ ]:
# ===========================================================================
#  CELL 16 — train one adapter per fold on the four training folds, then answer the held-out fold
#  with the plain zero-shot prompt. No examples in the prompt: whatever it knows, it learned.
# ===========================================================================
pred, ok_json, ok_schema, t0 = [None] * len(D), [], [], time.time()
for k in RUN_FOLDS:
    tr, te = FOLDS[k]; print(f"fold {k}: training on {len(tr)} reports")
    pm = lora_train(list(tr))
    for i in te:
        rec, vj, vs = parse(generate(pm, messages(TEXTS[i])))
        pred[i] = rec; ok_json.append(vj); ok_schema.append(vs)
    del pm; print(f"  fold {k} done, {time.time()-t0:.0f} s so far")
PRED["LoRA"], VALID["LoRA"] = pred, (np.mean(ok_json), np.mean(ok_schema))
print(f"valid JSON {100*np.mean(ok_json):.1f}% · every field valid {100*np.mean(ok_schema):.1f}%")
show("LoRA", score(pred, EVAL))
show("TF-IDF + LR, same folds", score(PRED["tfidf_lr"], EVAL))

**Results, all five folds.**

| | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted | mean | all fields valid |
|---|---|---|---|---|---|---|---|---|
| retrieved 3 | 0.492 | 0.667 | 0.686 | 0.697 | 0.327 | 0.528 | 0.566 | 95% |
| **LoRA** | 0.413 | 0.182 | 0.805 | 0.706 | 0.051 | 0.043 | **0.367** | 96.7% |
| *TF-IDF + LR* | 0.578 | 0.882 | 0.886 | 0.707 | 0.688 | 0.590 | *0.722* | — |

Against the bag of words, LoRA is within 0.01 on `injury`, and trails on the other 5. Against retrieval it wins on `vehicles`, `injury`
and loses everywhere else, so its mean (0.367) sits between fixed examples
(0.339) and retrieval (0.566).

**Fold by fold.** The last column is the answer loss in the final epoch of training.

| held-out fold | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted | mean | train loss |
|---|---|---|---|---|---|---|---|---|
| fold 0 (79 reports) | 0.203 | 0.000 | 0.303 | 0.624 | 0.000 | 0.000 | 0.188 | 0.064 |
| fold 1 (82 reports) | 0.482 | 0.154 | 0.829 | 0.714 | 0.000 | 0.174 | 0.392 | 0.041 |
| fold 2 (80 reports) | 0.398 | 0.000 | 0.867 | 0.769 | 0.500 | 0.000 | 0.422 | 0.051 |
| fold 3 (79 reports) | 0.382 | 0.211 | 0.881 | 0.747 | 0.000 | 0.000 | 0.370 | 0.051 |
| fold 4 (79 reports) | 0.545 | 0.667 | 0.864 | 0.680 | 0.000 | 0.000 | 0.459 | 0.040 |

Fold 0 is the cautionary tale. Its adapter scored at the majority-class level on `crash_type` and `vehicles`
(0.203 and 0.303, against 0.190 and 0.300 for always
guessing the commonest answer) and 0 on every yes/no field except injury, so its mean is 0.188. Yet its training
loss (0.064) looks as healthy as the others'. The other four folds learned
something real about `vehicles` and `crash_type`, and the spread between folds is wide. One adapter
per fold is not a luxury: a single run could have landed on fold 0's number, or fold
4's.

**What LoRA learned, and what it did not.**
- **The format.** 96.7% of replies had all six fields present and allowed, and
  that is with the plain zero-shot prompt that managed 70.9% before training.
- **The common, wordy fields.** `vehicles` reaches 0.805 macro-F1, the
  best of any language-model rung. On `crash_type` it is right 73.2% of the time, more often
  than retrieval (67.2%). Its macro-F1 is lower (0.413 against
  0.492) because it rarely says `pedestrian_bicycle` (7 times
  out of 399, where the corpus has 35) and never gets `other`. Accuracy rewards
  leaning on the big class. Macro-F1 does not.
- **Not the rare yes/no fields.** Across all 399 reports it said "yes" to pedestrian 8 times,
  to hit-and-run 2 times and to distracted 3 times. The corpus has
  36, 37 and 89 of each. It collapsed to "no",
  which keeps accuracy between 77% and 91% and is worth almost nothing in F1.

**Why the loss lied.** The answer loss averages over every answer token. Most of them are braces,
quotes and field names, identical in every reply and quickly learned. The few tokens that carry a rare
"yes" are a small share of that average. Two epochs at this learning rate was enough to make the
average small without making "yes" likely. The fix belongs in the training set, not the test fold:
ask the adapter about some of its own training reports (exercise 5). If it says "no" there too, it is
under-fitting the rare fields.

**What it cost.** On the build machine (8 CPU threads, bfloat16, no GPU), training the five adapters took
**313.6 minutes** in total, and answering all 399 held-out reports took
37.0 minutes (5.56 s per report). Per fold, training ranged from
12.8 to 159.0 minutes for the same amount of work, because the machine was shared, so read
the shortest folds as the better guide. 7 of about 800 optimiser steps were skipped as non-finite
(bfloat16 on a CPU; the T4 runs in float32). **TODO-after-run**: time the training and answering on a Colab T4 and
put the numbers here.

## 8 · One scoreboard, and what each line cost

In [ ]:
# ===========================================================================
#  CELL 17 — one scoreboard, same folds for everyone.
# ===========================================================================
order = ["tfidf_lr", "zero-shot", "few-shot, fixed 3", "few-shot, retrieved 3", "retrieved 3, LEAKY", "LoRA"]
print(f"folds {RUN_FOLDS} · {len(EVAL)} test reports · F1 (macro-F1 for crash_type and vehicles)\n")
print(f"{'':24s}" + "".join(f"{f[:11]:>12s}" for f in list(FIELDS) + ["mean"]) + f"{'valid':>8s}")
for name in [o for o in order if o in PRED]:
    s = score(PRED[name], EVAL)
    v = f"{100*VALID[name][1]:7.0f}%" if name in VALID else "      —"
    print(f"{name[:24]:24s}" + "".join(f"{s[f]['F1']:12.3f}" for f in list(FIELDS) + ["mean"]) + v)

All 399 reports, five corridor folds, the same scorer:

| Method | crash_type | pedestrian | vehicles | injury | hit_and_run | distracted | mean | JSON valid |
|---|---|---|---|---|---|---|---|---|
| Majority class | 0.190 | 0.000 | 0.300 | 0.721 | 0.000 | 0.000 | **0.202** | — |
| TF-IDF + logistic regression | 0.578 | 0.882 | 0.886 | 0.707 | 0.688 | 0.590 | **0.722** | — |
| Zero-shot prompt | 0.191 | 0.000 | 0.072 | 0.623 | 0.049 | 0.154 | **0.181** | 71% |
| Few-shot, 3 fixed examples | 0.370 | 0.262 | 0.585 | 0.721 | 0.053 | 0.042 | **0.339** | 99% |
| Few-shot, 3 retrieved (training corridors only) | 0.492 | 0.667 | 0.686 | 0.697 | 0.327 | 0.528 | **0.566** | 95% |
| LoRA fine-tune, zero-shot prompt | 0.413 | 0.182 | 0.805 | 0.706 | 0.051 | 0.043 | **0.367** | 97% |

Columns are F1 for the yes/no fields and macro-F1 for `crash_type` and `vehicles`. "JSON valid" is the
share of replies with all six fields present and allowed.

**Against DistilBERT from 2.13.** That model was fine-tuned one field at a time on the same folds. Its
F1 is at a 0.5 threshold; the language models give hard answers, so F1 is the only fair common number.
(Average precision needs a ranking, which a JSON answer does not have.)

| field | DistilBERT | TF-IDF + LR | retrieved 3 | LoRA |
|---|---|---|---|---|
| distracted | 0.453 | 0.590 | 0.528 | 0.043 |
| pedestrian | 0.914 | 0.882 | 0.667 | 0.182 |

**What it cost, on the build machine** (8 CPU threads, bfloat16, no GPU, shared with other jobs, so
treat the minutes as rough). **TODO-after-run**: T4 timings for every row.

| method | training | per report | all 399 reports |
|---|---|---|---|
| TF-IDF + LR | ~63 s for all fields and folds | milliseconds | — |
| zero-shot | none | 2.57 s | 17.1 min |
| fixed 3 | none | 2.29 s | 15.2 min |
| retrieved 3 | none | 1.96 s | 13.0 min |
| retrieved 3, leaky | none | 9.14 s | 60.8 min |
| LoRA | 313.6 min (5 adapters) | 5.56 s | 37.0 min |
| DistilBERT (2.13), one field | 35.1 min (distracted), 27.9 min (pedestrian) | — | — |

**Reading the board.** Nothing beats the bag of words. TF-IDF + LR averages 0.722 and is the best method on 5 of the six fields. Among the language-model rungs, the honest order is retrieved examples (0.566) > LoRA (0.367) > fixed examples (0.339) > zero-shot (0.181). Retrieval and LoRA fail in different places: retrieval is the better reader of rare yes/no fields, LoRA the better counter of vehicles. The language models also cost seconds per report where the bag of words costs milliseconds.

## 9 · The ugly cases

Aggregate counts only; the cell does not print narratives. Cell 19 lets you read one disagreement on
your own machine.

In [ ]:
# ===========================================================================
#  CELL 18 — the ugly cases, counted (no narrative text is printed).
# ===========================================================================
HR_RX = AUDIT["hit_and_run"][1]
hr = [i for i in EVAL if GOLD[i]["hit_and_run"] == "yes"]
said = [i for i in hr if re.search(HR_RX, TEXTS[i], re.I)]
silent = [i for i in hr if i not in said]
print(f"hit-and-run positives in these folds: {len(hr)} · narrative says so: {len(said)} · silent: {len(silent)}")
for name in [o for o in order if o in PRED]:
    rec = lambda ids: np.mean([(PRED[name][i] or {}).get("hit_and_run") == "yes" for i in ids]) if ids else float("nan")
    print(f"  {name[:24]:24s} recall when the narrative says so {rec(said):.2f} · when it is silent {rec(silent):.2f}")

print("\ncrash_type confusions (gold -> predicted), counts:")
for name in [o for o in order if o in PRED]:
    c = collections.Counter((GOLD[i]["crash_type"], (PRED[name][i] or {}).get("crash_type") or "invalid")
                            for i in EVAL)
    worst = [(f"{g}->{p}", n) for (g, p), n in c.most_common() if g != p][:4]
    print(f"  {name[:24]:24s}", worst)

**1 · The hit-and-run the narrative does not mention.** Of 37 coded hit-and-runs,
the narrative says so in about 23, going by the section 1 regex. The rest are ticked on
the form and never described. Recall split by that:

| | recall, narrative says so | recall, narrative silent |
|---|---|---|
| zero-shot | 0.04 | 0.00 |
| fixed 3 | 0.04 | 0.00 |
| retrieved 3 | 0.26 | 0.14 |
| LoRA | 0.04 | 0.00 |

A reader cannot be blamed for the second column. An evaluation that does not separate the two will
blame it anyway.

**2 · Left turn versus angle.** The coded crash type is a judgement about *which movement* caused the
crash. Narratives describe both vehicles' movements, often with the left-turner second. These are the
biggest off-diagonal cells, with pedestrian crashes called something else and the rare `other` class:

| | angle → left_turn | left_turn → angle | pedestrian/bicycle → a vehicle type | other, missed |
|---|---|---|---|---|
| retrieved 3 | 46 | 38 | 12 | 16 of 16 |
| LoRA | 36 | 15 | 28 | 16 of 16 |

**3 · `other` is a bin, not a class.** It holds 16 reports: rear ends, a head-on,
a sideswipe, an off-road. A macro-F1 counts it as a quarter of `crash_type`, so a few reports swing the
field.

**4 · Injury is a coded severity, not a sentence.** "Complained of pain" or "transported as a
precaution" may or may not be coded as an injury. Every method sits in a narrow band on `injury`, which
suggests the field itself is noisy against the text.

**5 · Format failures are real failures.** A reply that renames `pedestrian` to `pedestrian_bicycle` is
scored as wrong on that field. In production that reply breaks the parser, not just the metric.

> **Your turn — exercise 4.** Re-score every method with the "narrative silent" hit-and-runs removed
> from the evaluation (treat them as unknown, not as negatives). Which method's ranking changes?

In [ ]:
# ===========================================================================
#  CELL 19 (optional) — read ONE disagreement yourself. Clear this output before sharing the notebook.
# ===========================================================================
SHOW = False                                   # set True on your own machine only
name = "LoRA" if "LoRA" in PRED else "few-shot, retrieved 3"
wrong = [i for i in EVAL if (PRED[name][i] or {}).get("crash_type") != GOLD[i]["crash_type"]]
print(f"{len(wrong)} crash_type misses for {name}")
if SHOW and wrong:
    i = wrong[0]
    print("gold:", GOLD[i]["crash_type"], "· predicted:", (PRED[name][i] or {}).get("crash_type"))
    print(TEXTS[i][:300], "...")

## 10 · What this was for

**The answer to the lab's question.** **No, not at these settings.** On these 399 reports and these corridor folds, LoRA averages 0.367 across the six fields. Retrieval from training corridors averages 0.566, and a TF-IDF logistic regression trained in about a minute averages 0.722. LoRA learned the format and the common fields: it is the best language-model rung on `vehicles` and the most often right on `crash_type`. It collapsed to "no" on the rare yes/no fields, and on one fold it collapsed almost entirely, behind a training loss that looked fine. Retrieval wins the mean, but part of its gain is copying neighbours, and a leaky index inflates it further (section 6). Read that with the costs in section 8 and the split
in section 3. This is one small model, one prompt, and one untuned set of LoRA settings. It is
a fair answer to "does LoRA help out of the box", not to "can LoRA ever help". Exercise 5 is how you would
find out, without touching the test folds.

```
Fit card — LoRA on a small open language model, for text-to-fields extraction

Idea in one sentence:  Freeze a pretrained instruction-following model and train a thin,
                       low-rank correction so it answers your schema the way your coders do.

Why it works (the logic):  The model already reads English and writes JSON; what it lacks is
                       your field definitions and your coders' habits. Those are a small change
                       to how it maps text to answer, which a low-rank update can hold.

Fits these tasks:  extraction · classification into a fixed schema · several fields at once,
                       where one model answering a JSON record beats six separate classifiers to
                       maintain; data that cannot leave the agency, so a hosted API is out.

Needs this data:  a few hundred labelled reports, where the labels come from fields the text
                       actually supports (section 1); a split that holds out whole places, so the
                       score predicts a new corridor.

Breaks when:  the field is not in the text (lighting, weather, surface); the label is a
                       judgement the text leaves ambiguous (left turn vs angle); a field is rare
                       (1 in 10 to 1 in 5 here) and the adapter learns to say "no", while the average
                       loss still looks fine; one training run is trusted without fold-to-fold spread.

Keep the classical method when:  the fields are few and keyword-shaped, the budget is a CPU,
                       or you need a model you can audit term by term. On this corpus a TF-IDF
                       logistic regression trained in about a minute beat every rung of the ladder.
```

### Where this goes next, in the field

| Domain | Task | Data you would need | The baseline to beat |
|---|---|---|---|
| **Crash records (FDOT, HSMV)** | Flag reports where the coded fields disagree with the narrative, for review | the agency's own narratives and coded forms | a keyword audit and a TF-IDF classifier per field |
| **Safety analysis (HSIP, Vision Zero)** | Pull pedestrian and turning-movement detail that the form does not code | a few hundred reports coded by an analyst for the new field | the analyst reading reports, timed |
| **Transit operations** | Turn incident logs into structured cause codes | the agency's incident log and its cause taxonomy | the cause code the operator picked from a dropdown |
| **Agents (Module 6–7)** | A small local model as the extraction tool an agent calls | the same schema, plus checks on the tool's output | the same agent calling a prompted model |

### Evaluation close

- **Baseline.** Majority class, and TF-IDF + logistic regression with the 2.13 settings on the same
  folds. DistilBERT from 2.13 for the two fields it was trained on.
- **The split and why it is honest.** Five folds grouped by corridor, identical to 2.13. Retrieval
  examples come from training corridors only, and section 6 measures what happens otherwise. Nothing was
  tuned on test folds.
- **The ugly cases.** Coded fields that the narrative never mentions; left turn versus angle;
  a catch-all `other`; injury coding that the text does not settle; replies that break the schema.

### Exercises, collected

1. Widen the hit-and-run regex and re-run the availability audit (section 1).
2. Change the number of retrieved examples (section 6).
3. Build the nearest-neighbour copy baseline (section 6).
4. Re-score with the silent hit-and-runs removed (section 9).
5. **LoRA settings.** Try rank 4 and rank 64, or 1 epoch and 4, or a higher learning rate. First ask
   the adapter about 40 of its own *training* reports: if it gives the usual answer even there, it is
   under-fitting, and you learned that without touching the test fold. Pick the setting using fold 0's
   training corridors only (split them again by corridor), then report all five folds once.
6. Swap `GroupKFold` for `StratifiedGroupKFold` and report which fields move (section 3).
7. Swap in `Qwen/Qwen2.5-1.5B-Instruct`. It still fits on a T4 for inference; for LoRA you may need
   a smaller batch or shorter narratives. Does the zero-shot rung stop collapsing?

### Check yourself

1. Zero-shot replies were valid JSON almost every time. Why is that number nearly useless on its own,
   and what would you report next to it?
2. A colleague builds the retrieval index once on all 399 reports and then runs the folds.
   What exactly leaks, and which field in this lab would you expect to be inflated most?
3. An agency wants `light_condition` extracted from narratives. Using section 1, write the two sentences
   you say back.
4. LoRA tied the bag of words on `injury` and trailed it badly on `hit_and_run`, with a training loss
   that looked fine on every fold. Name one reason for each, and one check you could run on the
   training corridors alone that would have warned you.

---

*Built with `Qwen/Qwen2.5-0.5B-Instruct` (Apache-2.0), `transformers`, `peft` and `scikit-learn`. The
narratives are 399 redacted Florida HSMV long-form crash reports, distributed to enrolled
students through Canvas and not included in this notebook. Every number in the prose was produced by the
reproducer in `Labs/lora_crash/` on a CPU (no GPU). Your T4 run will differ slightly, and LoRA, which varied
most from fold to fold, may differ most. AI4Mobility.*